In [ ]:
%pip install yfinance
%pip install matplotlib
%pip install pandas
%pip install numpy
%pip install oracledb

In [ ]:
# import modules
from datetime import datetime
import yfinance as yf
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import oracledb
import typing
import re
import random

In [ ]:
# DB Stuff
ORACLE_HOST = "10.19.49.10"
ORACLE_PORT = 1522
ORACLE_SERVICE = "XE"
ORACLE_USER = "proiect"
ORACLE_PASSWORD = "proiect"

def db_get_connection() -> oracledb.Connection:
   """Se conecteaza la baza de date si returneaza conexiunea

   Returns:
      oracledb.Connection: Conexiunea la baza de date Oracle
   """
   dsn = f"{ORACLE_HOST}:{ORACLE_PORT}/{ORACLE_SERVICE}"
   print(f"Connecting to Oracle DB with DSN: '{dsn}'")
   connection: oracledb.Connection = oracledb.connect(user=ORACLE_USER, password=ORACLE_PASSWORD, dsn=dsn)
   print(f"Connected to Oracle DB with DSN: '{dsn}'; user: '{ORACLE_USER}'")
   return connection

def db_run_sql(conn: oracledb.Connection, 
               sql: str, 
               params: typing.Optional[dict] = None, 
               fetch: bool = False, 
               commit: bool = False,
               silent: bool = False) -> typing.Optional[list]: 
   """Executa o interogare SQL pe baza de date

   Args:
      conn (oracledb.Connection): Conexiunea la baza de date
      sql (str): Interogarea SQL de executat
      params (dict, optional): Parametrii pentru interogare. Defaults to None.
      fetch (bool, optional): Daca True, returneaza rezultatele interogarii. Defaults to False.
      commit (bool, optional): Daca True, face commit dupa executarea interogarii. Defaults to False.
      silent (bool, optional): Daca True, nu afiseaza interogarea SQL care se executa. Defaults to False.
      
   Returns:
      list: Rezultatele interogarii daca fetch este True, altfel None
   """
   if not silent:
      sql_print = str(sql).replace('\n', ' ').replace('\t', ' ')
      if len(sql_print) > 100:
         sql_print = sql_print[:100] + '...'
      print(f"Running SQL query: fetch={fetch}, commit={commit}, SQL: \"{sql_print}\", params: {params}")
   cursor = conn.cursor()
   if params:
      cursor.execute(sql, params)
   else:
      cursor.execute(sql)
   
   if fetch:
      return cursor.fetchall()

   if commit:
      conn.commit()
   return None

oracle_conn: oracledb.Connection = db_get_connection()

### Simulare ordine executie -> executie ordin o zi mai tarziu pentru o parte dintre ordine -> detinere portofoliu

In [ ]:
# simulare ordine executie
id_portofoliu_list: list[int] = [p[0] for p in db_run_sql(conn=oracle_conn, sql="SELECT id_portofoliu FROM portofoliu", fetch=True)] #type: ignore
ticker_list: list[str] = [s[0] for s in db_run_sql(conn=oracle_conn, sql="SELECT ticker FROM simbol_bursier", fetch=True)] #type: ignore

# Generare ordin pentru fiecare portofoliu, pentru fiecare simbol, pentru o data intre 2026-01-01 si 2026-04-30
ordine_executie_sql_inserts: list[str] = []
for id_portofoliu in id_portofoliu_list:
   print(f"\nGenerare ordine executie pentru portofoliu: {id_portofoliu}")
   
   for ticker in ticker_list:
      # print(f"  Simbol: {ticker}")
      data_ordin = datetime.strptime(f"2026-{random.randint(1,4):02d}-{random.randint(1,28):02d}", "%Y-%m-%d").date()
      sens_ordin = random.choice(['CUMPARARE', 'VANZARE'])
      tip_ordin = random.choice(['LIMITA', 'PIATA'])
      cantitate = random.randint(1, 100)
      status_ordin = random.choice(['NOU', 'NOU', 'PARTIAL', 'EXECUTAT', 'ANULAT'])
      
      
      preturi_deschidere: list[tuple[float]] = db_run_sql(conn=oracle_conn,
         sql="SELECT pret_deschidere FROM istoric_pret WHERE ticker = :ticker ORDER BY data_cotatie ASC",
         params={"ticker": ticker},
         fetch=True
      ) # type: ignore
      
      pret_limita: float = random.choice(preturi_deschidere)[0]
      pret_limita = pret_limita * (1 + random.uniform(-0.05, 0.05)) # adauga o variatie de +/- 5% la pretul de deschidere
      print(f"    Pret limita folosit: {pret_limita} - {type(pret_limita)}")
      
      sql_insert = f"""
         INSERT INTO ordin (id_portofoliu, ticker, tip_sens, tip_ordin, cantitate, pret_limita, data_ordin, status_ordin)
         VALUES ({id_portofoliu}, '{ticker}', '{sens_ordin}', '{tip_ordin}', {cantitate}, {pret_limita}, TO_DATE('{data_ordin}', 'YYYY-MM-DD'), '{status_ordin}');
      """.strip()
      sql_insert = re.sub(r"\s+", " ", sql_insert)
      ordine_executie_sql_inserts.append(sql_insert)
      
# shuffle ordin_executie_sql_inserts to select 10 random inserts
random.shuffle(ordine_executie_sql_inserts)
ordine_executie_sql_inserts = ordine_executie_sql_inserts[:30] # limitare la 30 insert-uri

print("\nGenerated SQL INSERT statements for 'ordin':")
for sql in ordine_executie_sql_inserts:
   print(sql)

In [ ]:
# In baza ordinelor de executie, simulare executie_ordin pentru 60% dintre ordinele in status NOU -> schimbare status la EXECUTAT
# Pentru fiecare ordin executat se va inregistra o executie ordin
ordine_noi: list[int] = [o[0] for o in db_run_sql(conn=oracle_conn, sql="SELECT id_ordin FROM ordin WHERE status_ordin = 'NOU'", fetch=True)] #type: ignore

# select random 60% dintre ordinele noi pentru a le schimba statusul la EXECUTAT
random.shuffle(ordine_noi)
ordine_noi = ordine_noi[:int(len(ordine_noi) * 0.6)]

print(f"\nSimulare executie ordin pentru {len(ordine_noi)} ordine noi (60% dintre ordinele noi):", ordine_noi)

ordine_sql_updates: list[str] = []
executie_ordin_sql_inserts: list[str] = []
detinere_portofoliu_sql_inserts: list[str] = []

for id_ordin in ordine_noi:
   print(f"\nSimulare executare ordin cu id: {id_ordin} ...")
   d = db_run_sql(
      conn=oracle_conn, 
      sql=f"SELECT ticker, tip_sens, tip_ordin, pret_limita, data_ordin FROM ordin WHERE id_ordin = {id_ordin}", 
      fetch=True, silent=True) #type: ignore
   ticker, tip_sens, tip_ordin, pret_limita, data_ordin = d[0] #type: ignore
   
   # obtinere id_bursa pentru ticker
   id_bursa = db_run_sql(
      conn=oracle_conn, 
      sql=f"SELECT id_bursa FROM simbol_bursier WHERE ticker = '{ticker}'", 
      fetch=True, silent=True)[0][0] #type: ignore
   
   cantitate_excutata: int = db_run_sql(
      conn=oracle_conn,
      sql=f"SELECT cantitate FROM ordin WHERE id_ordin = {id_ordin}",
      fetch=True, silent=True)[0][0] #type: ignore
   
   print(f"   Simulare pentru ordin: id_ordin={id_ordin}, ticker={ticker}, tip_sens={tip_sens}, tip_ordin={tip_ordin}, pret_limita={pret_limita}, data_ordin={data_ordin} (of type {type(data_ordin)}), id_bursa={id_bursa}, cantitate_executata={cantitate_excutata}")
   
   if tip_ordin == 'PIATA': 
      # Executare imediata, la aceeasi data ca data_ordin
      ordine_sql_updates.append(f"UPDATE ordin SET status_ordin = 'EXECUTAT' WHERE id_ordin = {id_ordin};")
      
      executie_ordin_sql_ins = f"""
         INSERT INTO executie_ordin (id_ordin, id_bursa, cantitate_executata, pret_executie, data_executie)
         VALUES ({id_ordin}, {id_bursa}, {cantitate_excutata}, {pret_limita}, TO_DATE('{data_ordin.strftime('%Y-%m-%d')}', 'YYYY-MM-DD'));
      """.strip()
      executie_ordin_sql_ins = re.sub(r"\s+", " ", executie_ordin_sql_ins)
      executie_ordin_sql_inserts.append(executie_ordin_sql_ins)
   elif tip_ordin == 'LIMITA':
      # Executare la 2 zile dupa data_ordin
      data_executie = data_ordin + pd.Timedelta(days=2)
      ordine_sql_updates.append(f"UPDATE ordin SET status_ordin = 'EXECUTAT' WHERE id_ordin = {id_ordin};")
      
      executie_ordin_sql_ins = f"""
         INSERT INTO executie_ordin (id_ordin, id_bursa, cantitate_executata, pret_executie, data_executie)
         VALUES ({id_ordin}, {id_bursa}, {cantitate_excutata}, {pret_limita}, TO_DATE('{data_executie.strftime('%Y-%m-%d')}', 'YYYY-MM-DD'))
      """.strip()
      executie_ordin_sql_ins = re.sub(r"\s+", " ", executie_ordin_sql_ins)
      executie_ordin_sql_inserts.append(executie_ordin_sql_ins)

print("\nGenerated SQL UPDATE statements for 'ordin':")
for sql in ordine_sql_updates:
   print(sql)
   
print("\nGenerated SQL INSERT statements for 'executie_ordin':")
for sql in executie_ordin_sql_inserts:
   print(sql)

In [ ]:
# Executare detinere portofoliu, in baza ordinelor executate, daca un portofoliu 
# are mai multe ordine executate pentru acelasi ticker, se va face o singura inregistrare in detinere portofoliu, cu media ponderata a preturilor
detinere_portofoliu_sql_inserts: list[str] = []

executii: list[tuple[int, int, float, float]] = db_run_sql(
   conn=oracle_conn,
   sql="SELECT id_ordin, id_bursa, cantitate_executata, pret_executie FROM executie_ordin",
   fetch=True) #type: ignore
for id_ordin, id_bursa, cantitate_executata, pret_executie in executii:
   print(f"Simulare detinere: id_ordin: {id_ordin}, id_bursa: {id_bursa}, cantitate_executata: {cantitate_executata}, pret_executie: {pret_executie}")
   
   id_portofoliu: int = db_run_sql(oracle_conn, f"SELECT id_portofoliu FROM ordin WHERE id_ordin = {id_ordin}", fetch=True)[0][0] #type: ignore
   ticker: str = db_run_sql(oracle_conn, f"SELECT ticker FROM ordin WHERE id_ordin = {id_ordin}", fetch=True)[0][0] #type: ignore
   cantitate: float = cantitate_executata
   pret_mediu: float = pret_executie # pentru simulare, consideram pretul de executie ca fiind pretul mediu de detinere
   data_actualizare: datetime = datetime.now()
   
   detinere_sql_ins = f"""
      INSERT INTO detinere_portofoliu (id_portofoliu, ticker, cantitate, pret_mediu, data_actualizare)
      VALUES ({id_portofoliu}, '{ticker}', {cantitate}, {pret_mediu}, TO_DATE('{data_actualizare.strftime('%Y-%m-%d')}', 'YYYY-MM-DD'))
   """.strip()
   detinere_sql_ins = re.sub(r"\s+", " ", detinere_sql_ins)
   detinere_portofoliu_sql_inserts.append(detinere_sql_ins)
print("\nGenerated SQL INSERT statements for 'detinere_portofoliu':")
for sql in detinere_portofoliu_sql_inserts:
   print(sql)